<a href="https://colab.research.google.com/github/coreprimejio/ev-server/blob/master-qa/Self_Contained_Quiz_PDF_Generator_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import math
import random
import uuid
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from pylatex import Document, Section, Command, Figure, MiniPage, LineBreak, Enumerate
from pylatex.utils import NoEscape, bold

# --- FIGURE GENERATION FUNCTIONS ---

def create_circular_division(total_parts, shaded_parts, output_filename):
    """Generates pie charts for circular divisions."""
    if total_parts <= 0: return

    num_wholes = shaded_parts // total_parts
    remaining_shaded = shaded_parts % total_parts

    num_charts = num_wholes + (1 if remaining_shaded > 0 else 0)
    if num_charts == 0: return

    fig, axes = plt.subplots(1, num_charts, figsize=(2.5 * num_charts, 2.5))
    if num_charts == 1: axes = [axes]

    for i in range(num_wholes):
        axes[i].pie([1], colors=['royalblue'], wedgeprops={'edgecolor': 'black', 'linewidth': 1})
        axes[i].axis('equal')

    if remaining_shaded > 0:
        sizes = [1] * total_parts
        colors = ['royalblue'] * remaining_shaded + ['lightgrey'] * (total_parts - remaining_shaded)
        axes[num_wholes].pie(sizes, colors=colors, startangle=90, counterclock=False,
                            wedgeprops={'edgecolor': 'black', 'linewidth': 1})
        axes[num_wholes].axis('equal')

    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=100)
    plt.close()

def create_rectangular_division(total_parts, shaded_parts, shading_style, output_filename):
    """Generates a horizontally long grid for rectangular divisions."""
    if total_parts == 0: return

    # Find factors to make the rectangle wider than it is tall
    factors = []
    for i in range(1, int(math.sqrt(total_parts)) + 1):
        if total_parts % i == 0:
            factors.append((i, total_parts // i))

    # Choose the factor pair with the biggest difference (most rectangular)
    rows, cols = min(factors, key=lambda x: abs(x[0] - x[1]))
    if rows > cols: # Ensure it's horizontally long
        rows, cols = cols, rows

    fig, ax = plt.subplots(figsize=(cols * 0.6, rows * 0.6))
    indices = list(range(total_parts))
    shaded_indices = random.sample(indices, shaded_parts)

    for i in range(total_parts):
        r, c = divmod(i, cols)
        is_shaded_part = i in shaded_indices

        if shading_style == 'half' and is_shaded_part:
            # Draw two distinct half-rectangles
            half1 = Rectangle((c, rows - 1 - r), 0.5, 1, facecolor='royalblue', edgecolor='black', linewidth=1)
            half2 = Rectangle((c + 0.5, rows - 1 - r), 0.5, 1, facecolor='lightgrey', edgecolor='black', linewidth=1)
            ax.add_patch(half1)
            ax.add_patch(half2)
        else:
            color = 'royalblue' if is_shaded_part else 'lightgrey'
            rect = Rectangle((c, rows - 1 - r), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect('equal', 'box')
    ax.axis('off')
    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=100)
    plt.close()

def create_triangular_division(total_parts, shaded_parts, shading_style, output_filename):
    """
    Generates an image of a large equilateral triangle divided into smaller,
    equal equilateral triangles, with a random subset being shaded.
    """
    divisions = int(math.sqrt(total_parts))
    if divisions * divisions != total_parts:
        print(f"Error: total_parts must be a perfect square. Got {total_parts}.")
        return

    if shaded_parts > total_parts:
        print(f"Error: shaded_parts cannot be greater than total_parts.")
        return

    side_length = 1.0
    height = side_length * math.sqrt(3) / 2.0
    all_triangles = []

    points = {}
    for row in range(divisions + 1):
        for col in range(row + 1):
            x = (col * side_length) - (row * side_length / 2.0)
            y = -row * height
            points[(row, col)] = (x, y)

    for row in range(divisions):
        for col in range(row + 1):
            p1 = points[(row, col)]
            p2 = points[(row + 1, col)]
            p3 = points[(row + 1, col + 1)]
            all_triangles.append([p1, p3, p2])

            if col < row:
                p1_up = points[(row, col)]
                p2_up = points[(row, col + 1)]
                p3_up = points[(row + 1, col + 1)]
                all_triangles.append([p1_up, p2_up, p3_up])

    indices = list(range(total_parts))
    shaded_indices = random.sample(indices, shaded_parts)

    fig, ax = plt.subplots(figsize=(5, 5))

    for i, vertices in enumerate(all_triangles):
        is_shaded_triangle = i in shaded_indices

        if shading_style == 'half' and is_shaded_triangle:
            p1, p2, p3 = vertices
            midpoint = ((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2)
            half1_vertices = [p1, midpoint, p3]
            half2_vertices = [midpoint, p2, p3]

            half1 = Polygon(half1_vertices, facecolor='royalblue', edgecolor='black', linewidth=1)
            half2 = Polygon(half2_vertices, facecolor='lightgrey', edgecolor='black', linewidth=1)
            ax.add_patch(half1)
            ax.add_patch(half2)
        else:
            color = 'royalblue' if is_shaded_triangle else 'lightgrey'
            triangle = Polygon(vertices, facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(triangle)

    ax.set_aspect('equal', 'box')
    ax.axis('off')

    main_triangle_base_width = divisions * side_length
    main_triangle_height = divisions * height
    plt.xlim(-main_triangle_base_width / 2 - 0.1, main_triangle_base_width / 2 + 0.1)
    plt.ylim(-main_triangle_height - 0.1, 0.1)

    plt.savefig(output_filename, format='jpg', bbox_inches='tight', pad_inches=0.1, dpi=100)
    plt.close()


# --- PDF GENERATION SCRIPT ---

def generate_quiz_pdf(quiz_data, pdf_filepath):
    """
    Reads a JSON object of questions and generates a single-column A4 PDF.
    """
    # 1. Setup Document
    geometry_options = {"tmargin": "1in", "lmargin": "1in"}
    # Use standard 'article' class for stability
    doc = Document(pdf_filepath, documentclass='article',
                   document_options=['a4paper', '12pt'],
                   geometry_options=geometry_options)

    # Add all necessary packages to the preamble
    doc.preamble.append(Command('usepackage', 'graphicx'))
    doc.preamble.append(Command('usepackage', 'amsmath'))
    doc.preamble.append(Command('usepackage', 'enumitem'))
    doc.preamble.append(Command('usepackage', 'xcolor')) # For colored text
    # Remove paragraph indentation
    doc.preamble.append(Command('setlength', [NoEscape(r'\parindent'), '0pt']))
    # Define a custom color for question headers
    doc.preamble.append(NoEscape(r'\definecolor{questioncolor}{RGB}{40,80,150}'))

    # Add Title and Author information
    doc.append(NoEscape(r'\begin{center}'))
    doc.append(NoEscape(r'{\Huge\bfseries Fractions Quiz}\\[10pt]'))
    doc.append(NoEscape(r'{\large Class 5}'))
    doc.append(NoEscape(r'\end{center}\bigskip'))

    # --- PATH CORRECTION ---
    # Get the directory where the PDF will be saved
    output_pdf_dir = os.path.dirname(pdf_filepath)
    # Define the figures directory relative to the PDF's location
    figures_dir = os.path.join(output_pdf_dir, 'figures')
    # Create the figures directory if it doesn't exist
    os.makedirs(figures_dir, exist_ok=True)

    # 2. Iterate through questions and add them to the PDF
    for i, q in enumerate(quiz_data['questions']):
        # Use a colored, bold header for each question
        doc.append(NoEscape(r'\textcolor{questioncolor}{\textbf{Question ' + str(i+1) + r':}} '))
        doc.append(NoEscape(q['question']))
        doc.append(NoEscape(r'\par')) # Add a paragraph break

        # Generate and add figure if it exists
        if 'figure_desc' in q:
            fig_desc = q['figure_desc']
            # Define the filename and the full path for saving the image
            fig_name_only = f'q_{i+1}.jpg'
            fig_full_path = os.path.join(figures_dir, fig_name_only)

            if fig_desc['type_of_division'] == 'circular':
                create_circular_division(fig_desc['total_parts'], fig_desc['shaded_parts'], fig_full_path)
            elif fig_desc['type_of_division'] == 'rectangular':
                create_rectangular_division(
                    fig_desc['total_parts'],
                    fig_desc['shaded_parts'],
                    fig_desc.get('shading_style', 'full'), # Default to 'full'
                    fig_full_path
                )
            elif fig_desc['type_of_division'] == 'triangular':
                create_triangular_division(
                    fig_desc['total_parts'],
                    fig_desc['shaded_parts'],
                    fig_desc['shading_style'],
                    fig_full_path
                )

            # Add the figure to the PDF using a relative path
            if os.path.exists(fig_full_path):
                # The relative path from the .tex file is just 'figures/q_... .jpg'
                fig_relative_path = os.path.join('figures', fig_name_only)
                doc.append(NoEscape(r'\begin{center}'))
                doc.append(NoEscape(r'\includegraphics[width=0.3\linewidth]{' + fig_relative_path.replace('\\', '/') + '}'))
                doc.append(NoEscape(r'\end{center}'))

        # Add options
        with doc.create(Enumerate(options=NoEscape(r'label=(\alph*)'))) as enum:
            for option in q['options']:
                enum.add_item(NoEscape(option))

        # Add a horizontal line and spacing between questions
        doc.append(NoEscape(r'\bigskip\hrule\bigskip'))

    # 3. Generate PDF
    try:
        doc.generate_pdf(clean_tex=True)
        print(f"✅ Successfully generated PDF: {pdf_filepath}.pdf")
    except Exception as e:
        print(f"❌ PDF generation failed. Ensure a LaTeX distribution is installed. Error: {e}")

if __name__ == '__main__':
    # Define the input and output directories
    input_dir = '/Users/ajaygupta/class5/fractions/json'
    output_dir = '/Users/ajaygupta/class5/fractions/pdfs'

    # Construct the full path to the input JSON file
    json_file_path = os.path.join(input_dir, 'fractions.json')

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Generate a unique filename for the PDF
    pdf_filename = f"fraction-{uuid.uuid4()}"
    pdf_full_path = os.path.join(output_dir, pdf_filename)

    try:
        # Load the quiz data from the specified JSON file
        with open(json_file_path, 'r') as f:
            quiz_json_object = json.load(f)

        # Call the generator function with the loaded data and new file path
        generate_quiz_pdf(quiz_json_object, pdf_full_path)

    except FileNotFoundError:
        print(f"❌ Error: The file was not found at {json_file_path}")
    except json.JSONDecodeError:
        print(f"❌ Error: The file at {json_file_path} is not a valid JSON file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

✅ Successfully generated PDF: /Users/ajaygupta/class5/fractions/pdfs/fraction-f328fae3-93ae-49e1-a8f9-caac1c3b8379.pdf
